# claim-bevplanner-nuscenes-straight-driving-imbalance 统计复算

论文声称（尚未独立验证）：nuScenes 73.9% 的数据是直行场景（§1/Fig.2）；附录 E：87% of evaluation samples。

**预注册（详见仓库 README，不得回调）**：
- 判据族：H=累计|Δheading| ∈ {3,5,10,15,20,30}°；L=横向偏移 ∈ {0.5,1,2,3} m；P=逐点|Δheading|全<5°
- 主分母：完整未来轨迹样本（未来≥6 keyframe @2Hz，horizon 3s）；被截断样本单列
- 触发 falsifier 1（证伪级）：全部判据×阈值×split 占比均 <60%
- 带内复得（证据级）：trainval 完整样本级某判据占比 ∈ [68.9%,78.9%]
- 附录口径复得（证据级）：val 占比 ∈ [82%,92%]（6019 与 5119 两分母各一行）；旁证：val 完整样本/6019 ≈ 85%±2pp（对 5119）
- 确定性断言：scene-0001 重算两次逐位一致；全量普查无抽样随机性
- git_commit: c24715a

In [ ]:
import json, os, sys, time, math, csv, glob, platform, hashlib, subprocess, urllib.request

CLAIM_ID = 'claim-bevplanner-nuscenes-straight-driving-imbalance'
GIT_COMMIT = 'c24715a'
T0 = time.time()
OUT = '/kaggle/working'
os.makedirs(OUT, exist_ok=True)

H_THRESH = [3, 5, 10, 15, 20, 30]
L_THRESH = [0.5, 1, 2, 3]
P_THRESH = 5.0
HORIZON = 6
ANCHORS = {'scenes': 850, 'samples': 34149}

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

cpu_model = ''
for line in open('/proc/cpuinfo'):
    if line.startswith('model name'):
        cpu_model = line.split(':', 1)[1].strip()
        break
ram_gb = round(int(open('/proc/meminfo').readline().split()[1]) / 1e6, 1)
ENV = {'cpu_model': cpu_model, 'ram_gb': ram_gb, 'python': sys.version.split()[0],
       'platform': platform.platform()}
print('ENV:', ENV)
print('start:', time.strftime('%Y-%m-%dT%H:%M:%S', time.gmtime()))

In [ ]:
# 定位元数据四件套 + 完整性门（对预注册锚点，>1% 偏差即中止）
META = None
for r in sorted(glob.glob('/kaggle/input/*/')):
    for sub in ('', 'v1.0-trainval/'):
        base = r + sub
        if all(os.path.exists(base + f) for f in ('scene.json', 'sample.json', 'sample_data.json', 'ego_pose.json')):
            META = base
            break
    if META:
        break
print('meta_dir =', META)
assert META is not None, 'META_NOT_FOUND: 四件套不在任何挂载数据集内'

FILES = ['scene.json', 'sample.json', 'sample_data.json', 'ego_pose.json']
SIZES = {f: os.path.getsize(META + f) for f in FILES}
print('sizes:', SIZES)

t = time.time(); SCENES = json.load(open(META + 'scene.json')); print('scene.json loaded', round(time.time()-t, 1))
t = time.time(); SAMPLES = json.load(open(META + 'sample.json')); print('sample.json loaded', round(time.time()-t, 1))
t = time.time(); SDATA = json.load(open(META + 'sample_data.json')); print('sample_data.json loaded', round(time.time()-t, 1))
t = time.time(); EPOS = json.load(open(META + 'ego_pose.json')); print('ego_pose.json loaded', round(time.time()-t, 1))

COUNTS = {'scenes': len(SCENES), 'samples': len(SAMPLES), 'sample_data': len(SDATA), 'ego_pose': len(EPOS)}
ANCHORS_OK = (abs(COUNTS['scenes'] - ANCHORS['scenes']) / ANCHORS['scenes'] <= 0.01
              and abs(COUNTS['samples'] - ANCHORS['samples']) / ANCHORS['samples'] <= 0.01)
print('COUNTS:', COUNTS, 'anchors_ok:', ANCHORS_OK)
assert ANCHORS_OK, 'INTEGRITY_FAIL: 计数偏离官方锚点 >1%，停止判读'

PIP_FREEZE = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
open(OUT + '/pip-freeze.txt', 'w').write(PIP_FREEZE)
print('pip freeze lines:', len(PIP_FREEZE.splitlines()))

In [ ]:
# devkit splits.py（pinned commit，ast 解析不执行远端代码）
hdr = {'User-Agent': 'ka-verify'}
meta = json.load(urllib.request.urlopen(urllib.request.Request(
    'https://api.github.com/repos/nutonomy/nuscenes-devkit/commits/master', headers=hdr), timeout=30))
DEVKIT_SHA = meta['sha']
print('devkit commit:', DEVKIT_SHA)
url = f'https://raw.githubusercontent.com/nutonomy/nuscenes-devkit/{DEVKIT_SHA}/python-sdk/nuscenes/utils/splits.py'
src = urllib.request.urlopen(urllib.request.Request(url, headers=hdr), timeout=30).read().decode()

import ast
LISTS = {}
for node in ast.parse(src).body:
    if isinstance(node, ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], ast.Name) \
            and node.targets[0].id in ('train', 'val'):
        LISTS[node.targets[0].id] = ast.literal_eval(node.value)
TRAIN_SCENES = set(LISTS['train'])
VAL_SCENES = set(LISTS['val'])
print('train scenes:', len(TRAIN_SCENES), 'val scenes:', len(VAL_SCENES))

In [ ]:
# 构建 per-sample ego 未来轨迹（LIDAR_TOP keyframe → ego_pose）
def quat_yaw(q):
    w, x, y, z = q['w'], q['x'], q['y'], q['z']
    return math.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))

def wrap(a):
    return (a + math.pi) % (2 * math.pi) - math.pi

sample_by_token = {s['token']: s for s in SAMPLES}
sd_by_token = {d['token']: d for d in SDATA}
ep_by_token = {e['token']: e for e in EPOS}

lidar_kf = {}
for d in SDATA:
    if d.get('is_key_frame') and d.get('channel') == 'LIDAR_TOP':
        lidar_kf[d['sample_token']] = d['token']

RECORDS = []
EXCLUDED_NO_LIDAR = 0
t = time.time()
for sc in SCENES:
    tok = sc['first_sample_token']
    chain = []
    while tok:
        chain.append(tok)
        tok = sample_by_token[tok].get('next')
    split = 'val' if sc['name'] in VAL_SCENES else ('train' if sc['name'] in TRAIN_SCENES else 'other')
    for tok in chain:
        sd_tok = lidar_kf.get(tok)
        if sd_tok is None:
            EXCLUDED_NO_LIDAR += 1
            continue
        ep = ep_by_token[sd_by_token[sd_tok]['ego_pose_token']]
        p0 = ep['translation']
        fut_x, fut_y, fut_yaw = [], [], []
        nx = sample_by_token[tok].get('next')
        k = 0
        while nx and k < HORIZON:
            sd2 = lidar_kf.get(nx)
            if sd2 is not None:
                ep2 = ep_by_token[sd_by_token[sd2]['ego_pose_token']]
                fut_x.append(ep2['translation'][0])
                fut_y.append(ep2['translation'][1])
                fut_yaw.append(quat_yaw(ep2['rotation']))
                k += 1
            nx = sample_by_token[nx].get('next')
        RECORDS.append({'sample_token': tok, 'scene': sc['name'], 'split': split, 'n_future': k,
                        'x0': p0[0], 'y0': p0[1], 'yaw0': quat_yaw(ep['rotation']),
                        'fut_x': fut_x, 'fut_y': fut_y, 'fut_yaw': fut_yaw})
print('records:', len(RECORDS), 'excluded_no_lidar:', EXCLUDED_NO_LIDAR, 'in', round(time.time()-t, 1), 's')
from collections import Counter
print('split counts:', Counter(r['split'] for r in RECORDS))
print('n_future dist:', Counter(min(r['n_future'], 6) for r in RECORDS))

In [ ]:
# 判据计算 + 位移交叉校验 + 确定性断言
def sample_metrics(r):
    k = r['n_future']
    if k < 1:
        return {'cum_dh': None, 'max_dh': None, 'lateral': None, 'all_lt5': None}
    dhs = [abs(wrap(r['fut_yaw'][0] - r['yaw0']))]
    dhs += [abs(wrap(r['fut_yaw'][j] - r['fut_yaw'][j-1])) for j in range(1, k)]
    dx = r['fut_x'][-1] - r['x0']
    dy = r['fut_y'][-1] - r['y0']
    nx_, ny_ = -math.sin(r['yaw0']), math.cos(r['yaw0'])
    return {'cum_dh': sum(dhs), 'max_dh': max(dhs),
            'lateral': abs(dx * nx_ + dy * ny_),
            'all_lt5': max(dhs) < math.radians(P_THRESH)}

def disp_check(r):
    pts = [(r['x0'], r['y0'])] + list(zip(r['fut_x'], r['fut_y']))
    yaws = [r['yaw0']] + list(r['fut_yaw'])
    bad = tot = 0
    for j in range(1, len(pts)):
        dx = pts[j][0] - pts[j-1][0]
        dy = pts[j][1] - pts[j-1][1]
        if math.hypot(dx, dy) > 0.3:
            tot += 1
            if abs(wrap(math.atan2(dy, dx) - yaws[j-1])) > math.radians(15):
                bad += 1
    return bad, tot

t = time.time()
for r in RECORDS:
    r['m'] = sample_metrics(r)
    r['disp_bad'], r['disp_tot'] = disp_check(r)
print('metrics computed in', round(time.time()-t, 1), 's')

# 确定性断言：scene-0001 重算两次逐位一致
probe = [r for r in RECORDS if r['scene'] == 'scene-0001']
snap1 = [(r['sample_token'], r['m']['cum_dh'], r['m']['lateral']) for r in probe]
for r in probe:
    r['m'] = sample_metrics(r)
snap2 = [(r['sample_token'], r['m']['cum_dh'], r['m']['lateral']) for r in probe]
DETERMINISM_OK = snap1 == snap2
print('determinism_ok:', DETERMINISM_OK, f'({len(probe)} samples of scene-0001)')

bad_tot = sum(r['disp_tot'] for r in RECORDS)
bad_cnt = sum(r['disp_bad'] for r in RECORDS)
mismatched_samples = sum(1 for r in RECORDS if r['disp_tot'] > 0 and r['disp_bad'] / r['disp_tot'] > 0.5)
DISAGREE = {'steps_total': bad_tot, 'steps_conflict': bad_cnt,
            'samples_majority_conflict': mismatched_samples,
            'sample_rate': round(mismatched_samples / len(RECORDS), 5)}
print('displacement cross-check:', DISAGREE)

In [ ]:
# 阈值扫描（全部判据×阈值×分母，防择优：全表输出）
def ratios(pop):
    n = len(pop)
    res = {'n': n}
    if n == 0:
        return res
    for h in H_THRESH:
        res[f'H_{h}deg'] = round(sum(1 for r in pop if r['m']['cum_dh'] is not None
                                     and math.degrees(r['m']['cum_dh']) < h) / n, 5)
    for l in L_THRESH:
        res[f'L_{l}m'] = round(sum(1 for r in pop if r['m']['lateral'] is not None
                                   and r['m']['lateral'] < l) / n, 5)
    res['P_step_lt5deg'] = round(sum(1 for r in pop if r['m']['all_lt5']) / n, 5)
    return res

POPS = {
    'trainval_complete': [r for r in RECORDS if r['n_future'] >= HORIZON],
    'train_complete': [r for r in RECORDS if r['n_future'] >= HORIZON and r['split'] == 'train'],
    'val_complete': [r for r in RECORDS if r['n_future'] >= HORIZON and r['split'] == 'val'],
    'val_all6019_secondary': [r for r in RECORDS if r['split'] == 'val'],
}
RESULTS = {name: ratios(pop) for name, pop in POPS.items()}
for name, res in RESULTS.items():
    print(name, res)

# 场景级（背景，不参与判定）
scene_dh = {}
for r in RECORDS:
    if r['m']['cum_dh'] is not None:
        scene_dh.setdefault(r['scene'], []).append(math.degrees(r['m']['cum_dh']))
SCENE_LEVEL = {}
for h in H_THRESH:
    straight = sum(1 for v in scene_dh.values() if sum(v) / len(v) < h)
    SCENE_LEVEL[f'H_{h}deg'] = {'n_scenes': len(scene_dh), 'ratio': round(straight / len(scene_dh), 5)}
print('scene_level:', SCENE_LEVEL)

val_comp_n = RESULTS['val_complete']['n']
VAL_RATIO_6019 = round(val_comp_n / 6019, 5)
print(f'val complete-future: {val_comp_n}/6019 = {VAL_RATIO_6019} (5119 旁证带 0.8304–0.8704: {0.8304 <= VAL_RATIO_6019 <= 0.8704})')

In [ ]:
# 输出 + 三行结论摘要
summary = {
    'claim_id': CLAIM_ID, 'git_commit': GIT_COMMIT,
    'kaggle_notebook_version': 'FILLED_BY_HARVESTER',
    'env': ENV, 'env_pip_freeze_sha256': sha256_file(OUT + '/pip-freeze.txt'),
    'data': {'dataset': 'sahangunasekara92/nuscenes-v1-0-full-keyframes', 'meta_dir_suffix': META.replace('/kaggle/input/', '', 1),
             'sizes': SIZES, 'counts': COUNTS, 'anchors_ok': ANCHORS_OK,
             'excluded_no_lidar_keyframe': EXCLUDED_NO_LIDAR},
    'splits_sha': DEVKIT_SHA,
    'determinism_ok': DETERMINISM_OK,
    'displacement_cross_check': DISAGREE,
    'complete_future': {'trainval': RESULTS['trainval_complete']['n'], 'train': RESULTS['train_complete']['n'],
                        'val': val_comp_n, 'val_ratio_vs_6019': VAL_RATIO_6019,
                        'val_ratio_in_5119_band': 0.8304 <= VAL_RATIO_6019 <= 0.8704},
    'populations': RESULTS, 'scene_level_secondary': SCENE_LEVEL,
    'preregistered': {'H': H_THRESH, 'L': L_THRESH, 'P': P_THRESH, 'horizon': HORIZON,
                      'falsifier_line': 0.60, 'inband': [0.689, 0.789], 'appendix_band': [0.82, 0.92]},
    'runtime_sec': round(time.time() - T0, 1),
    'notes': '主分母=完整未来轨迹样本；val_all6019_secondary 含被截断样本（短 horizon 偏直行），仅敏感性参考不作判定。',
}
json.dump(summary, open(OUT + '/results.json', 'w'), indent=1, ensure_ascii=False)

with open(OUT + '/straight_ratio.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['population', 'criterion', 'threshold', 'n', 'ratio'])
    for name, res in RESULTS.items():
        for key, val in res.items():
            if key == 'n':
                continue
            crit = key.split('_')[0]
            thr = key[len(crit) + 1:]
            w.writerow([name, crit, thr, res['n'], val])

all_ratios = [v for name, res in RESULTS.items() if not name.endswith('secondary')
              for k, v in res.items() if k != 'n']
gmin = min(all_ratios)
trig = gmin < 0.60
inband = [f'{k}={v}' for k, v in RESULTS['trainval_complete'].items()
          if k != 'n' and 0.689 <= v <= 0.789]
appx = {k: v for k, v in RESULTS['val_complete'].items() if k != 'n' and 0.82 <= v <= 0.92}

L1 = f'line1 falsifier1_triggered={"YES" if trig else "NO"} (complete-population global min ratio={gmin})'
L2 = f'line2 in_band_trainval_complete={inband if inband else "NONE"}'
L3 = f'line3 appendix_band_val_complete={appx if appx else "NONE"} val_complete_n={val_comp_n}/6019={VAL_RATIO_6019}'
open(OUT + '/summary.txt', 'w').write('\n'.join([L1, L2, L3]))
print(L1)
print(L2)
print(L3)
print('DONE total_sec', round(time.time() - T0, 1))